# 03 — Zero-Shot Baseline

Runs Cosmos-Reason2-2B **without any fine-tuning** on a balanced eval set to establish baseline performance.

**Run order:** Cell 1 (install) → Cell 2 (mount + login) → Cell 3 (load val data) → Cell 4 (build eval set) → Cell 5 (load model) → Cell 6 (helpers) → Cell 7 (run inference) → Cell 8 (score) → Cell 9 (save analysis)

## Cell 1 — Install

In [ ]:
!pip install transformers accelerate decord fiftyone -q

## Cell 2 — Mount Drive & Login

In [ ]:
from google.colab import drive, userdata
from huggingface_hub import login

drive.mount('/content/drive', force_remount=True)
login(token=userdata.get("HF_TOKEN"))

import json, os, torch, numpy as np
from PIL import Image
import decord

# ── Paths ──────────────────────────────────────────────────────────────────
VAL_PATH      = "/content/drive/MyDrive/cosmos-cookoff/data/underwater_vqa_val.json"
BASELINE_PATH = "/content/drive/MyDrive/cosmos-cookoff/outputs/zero_shot_results/baseline.json"
os.makedirs(os.path.dirname(BASELINE_PATH), exist_ok=True)

MODEL_NAME    = "nvidia/Cosmos-Reason2-2B"
N_FRAMES      = 8
SYSTEM_PROMPT = (
    "You are a helpful assistant specializing in underwater scene understanding "
    "for autonomous underwater vehicle (AUV) navigation."
)

print(f"VAL_PATH exists: {os.path.exists(VAL_PATH)}")

## Cell 3 — Load validation data

In [ ]:
with open(VAL_PATH) as f:
    val_data = json.load(f)

print(f"Val set: {len(val_data)} pairs")
print(f"Sample: {val_data[0]['conversations'][0]['value'][:100]}")

## Cell 4 — Restore videos from HuggingFace & build balanced eval set

Downloads the WebUOT-238-Test videos to `/root/fiftyone/` and patches val data paths accordingly.

In [ ]:
import fiftyone as fo
from fiftyone.utils.huggingface import load_from_hub
import random

random.seed(42)

# Re-download videos — stored at /root/fiftyone/.../data/
fo_dataset = load_from_hub("Voxel51/WebUOT-238-Test")
FIFTYONE_VIDEO_DIR = "/root/fiftyone/huggingface/hub/Voxel51/WebUOT-238-Test/data"
print(f"Videos at: {FIFTYONE_VIDEO_DIR}")

def fix_video_path(path):
    """Remap Drive paths to local fiftyone paths."""
    return os.path.join(FIFTYONE_VIDEO_DIR, os.path.basename(path))

# Build balanced eval set: 5 samples per QA type
def filter_by_type(data, tag, n=5):
    subset = [x for x in data if tag in x["id"]]
    return random.sample(subset, min(n, len(subset)))

eval_samples = (
    filter_by_type(val_data, "_obj_id",   n=5) +
    filter_by_type(val_data, "_scene",    n=5) +
    filter_by_type(val_data, "_hazard",   n=5) +
    filter_by_type(val_data, "_spatial",  n=5) +
    filter_by_type(val_data, "_mcq_",     n=5) +
    filter_by_type(val_data, "_cat_mcq_", n=5)
)
random.shuffle(eval_samples)

# Patch video paths to local fiftyone copies
for s in eval_samples:
    s["video"] = fix_video_path(s["video"])

# Verify at least one video exists
print(f"Eval set: {len(eval_samples)} samples")
print(f"First video exists: {os.path.exists(eval_samples[0]['video'])}")
print(f"Sample path: {eval_samples[0]['video']}")

## Cell 5 — Load Cosmos-Reason2-2B (zero-shot, no adapter)

In [ ]:
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

print("Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
processor.image_processor.min_pixels = 256 * 28 * 28
processor.image_processor.max_pixels = 512 * 28 * 28

print("Loading model...")
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print(f"Model loaded on: {next(model.parameters()).device}")
print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

## Cell 6 — Helpers

In [ ]:
def extract_frames(video_path, n_frames=N_FRAMES):
    vr = decord.VideoReader(video_path, ctx=decord.cpu(0))
    indices = np.linspace(0, len(vr) - 1, n_frames, dtype=int)
    frames = vr.get_batch(indices).asnumpy()
    return [Image.fromarray(frames[i]) for i in range(n_frames)]


def run_inference(sample, n_frames=N_FRAMES, max_new_tokens=256):
    """Run zero-shot inference on a single video QA sample."""
    video_path = sample["video"]
    question   = sample["conversations"][0]["value"].replace("<video>\n", "").strip()

    try:
        frames = extract_frames(video_path, n_frames)
    except Exception as e:
        return {"error": str(e), "prediction": None}

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": [
            *[{"type": "image", "image": f} for f in frames],
            {"type": "text",  "text": question},
        ]},
    ]
    text   = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=frames, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    generated = out_ids[0][inputs["input_ids"].shape[1]:]
    prediction = processor.decode(generated, skip_special_tokens=True).strip()

    return {
        "id":           sample["id"],
        "video":        video_path,
        "question":     question,
        "ground_truth": sample["conversations"][1]["value"],
        "prediction":   prediction,
    }


print("Helpers ready.")

## Cell 7 — Run zero-shot inference

In [ ]:
from tqdm import tqdm

results = []
errors  = []

for sample in tqdm(eval_samples, desc="Zero-shot inference"):
    result = run_inference(sample)
    if "error" in result:
        errors.append(result)
        print(f"  ERROR on {sample['id']}: {result['error']}")
    else:
        results.append(result)
        print(f"\n[{result['id']}]")
        print(f"Q:    {result['question'][:80]}")
        print(f"GT:   {result['ground_truth'][:100]}")
        print(f"PRED: {result['prediction'][:150]}")
        print("-" * 60)

print(f"\nCompleted: {len(results)} OK, {len(errors)} errors")

## Cell 8 — Score results & save baseline

In [ ]:
def score_mcq(result):
    """Returns True if prediction contains the correct MCQ letter."""
    gt   = result["ground_truth"].strip().upper()
    pred = result["prediction"].strip().upper()
    correct_letter = gt[0]  # "A", "B", "C", or "D"
    return correct_letter in pred[:10]

mcq_results  = [r for r in results if "_mcq_" in r["id"] or "_cat_mcq_" in r["id"]]
open_results = [r for r in results if "_mcq_" not in r["id"] and "_cat_mcq_" not in r["id"]]

mcq_correct = sum(score_mcq(r) for r in mcq_results)
mcq_total   = len(mcq_results)

print("=" * 50)
print("ZERO-SHOT BASELINE RESULTS")
print("=" * 50)
print(f"MCQ accuracy:       {mcq_correct}/{mcq_total} = {mcq_correct/mcq_total*100:.1f}%")
print(f"Open-ended samples: {len(open_results)}")

# Save baseline results to Drive
baseline_output = {
    "model":        MODEL_NAME,
    "n_frames":     N_FRAMES,
    "eval_samples": len(results),
    "mcq_accuracy": f"{mcq_correct/mcq_total*100:.1f}%",
    "results":      results,
}
with open(BASELINE_PATH, "w") as f:
    json.dump(baseline_output, f, indent=2)
print(f"\nSaved to {BASELINE_PATH}")

## Cell 9 — Save qualitative analysis

In [ ]:
analysis = {
    "zero_shot_findings": {
        "mcq_accuracy":          "20.0% (below random chance for 4-choice MCQ)",
        "visibility_assessment": "Consistently wrong — model defaults to 'clear' regardless of actual conditions",
        "hazard_detection":      "Fundamentally misunderstood — describes animals as hazards instead of visual/tracking challenges",
        "spatial_reasoning":     "Partially correct on position, wrong on motion direction",
        "object_identification": "Strongest zero-shot capability — species recognition mostly correct",
        "conclusion":            "Model has no domain-specific understanding of underwater tracking challenges or AUV navigation context",
    }
}

ANALYSIS_PATH = "/content/drive/MyDrive/cosmos-cookoff/outputs/zero_shot_results/analysis.json"
with open(ANALYSIS_PATH, "w") as f:
    json.dump(analysis, f, indent=2)
print(f"Saved analysis to {ANALYSIS_PATH}")